# Split Strategy and Leakage Control

This notebook defines the modeling target, separates features from non-modeling columns, checks for leakage risks, and creates reproducible train, validation, and test splits.

Splitting must happen before any fitted preprocessing step such as imputation, encoding, scaling, capping, or feature selection.

## 1. Environment Setup

Load the required libraries and connect the notebook to the project source code.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

project_root = Path.cwd().resolve().parents[0]
src_dir = project_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from credit_risk_platform.data.german_credit import interim_german_credit_dir
from credit_risk_platform.utils.io import load_csv

RANDOM_STATE = 42

print("project_root:", project_root)
print("src_dir exists:", src_dir.exists())

project_root: /Users/ememakpan/Documents/New project/applied_ai_economics_portfolio/project_01_ai_credit_risk_platform
src_dir exists: True


## 2. Load the Modeling Dataset

Load the standardized dataset created during ingestion. This is still a raw modeling table, not a preprocessed modeling matrix.

In [2]:
interim_dir = interim_german_credit_dir(project_root)
df = load_csv(interim_dir / "german_credit_standardized.csv")

print("Shape:", df.shape)
df.head()

Shape: (1000, 24)


,applicant_id,checking_account_status,duration_months,credit_history,purpose,credit_amount,savings_account,employment_duration,installment_rate_pct_income,personal_status_sex,...,other_installment_plans,housing,existing_credits_count,job_type,liable_people_count,telephone,foreign_worker,risk_class,risk_label,TARGET
0,1,A11,6,A34,A43,1169,A65,A75,4,A93,...,A143,A152,2,A173,1,A192,A201,1,good,0
1,2,A12,48,A32,A43,5951,A61,A73,2,A92,...,A143,A152,1,A173,1,A191,A201,2,bad,1
2,3,A14,12,A34,A46,2096,A61,A74,2,A93,...,A143,A152,1,A172,2,A191,A201,1,good,0
3,4,A11,42,A32,A42,7882,A61,A74,2,A93,...,A143,A153,1,A173,2,A191,A201,1,good,0
4,5,A11,24,A33,A40,4870,A61,A73,3,A93,...,A143,A153,2,A173,2,A191,A201,2,bad,1


## 3. Define Target and Non-Modeling Columns

`TARGET` is the modeling target. Identifier and target-label columns are kept for traceability and interpretation but excluded from model features.

In [3]:
target_col = "TARGET"

identifier_cols = ["applicant_id"]
target_label_cols = ["risk_class", "risk_label"]
exclude_from_features = identifier_cols + target_label_cols + [target_col]

feature_cols = [col for col in df.columns if col not in exclude_from_features]

split_config = {
    "target_col": target_col,
    "identifier_cols": identifier_cols,
    "target_label_cols": target_label_cols,
    "excluded_from_features": exclude_from_features,
    "n_features": len(feature_cols),
}

display(pd.DataFrame({"feature_cols": pd.Series(feature_cols)}))
split_config

,feature_cols
0,checking_account_status
1,duration_months
2,credit_history
3,purpose
4,credit_amount
5,savings_account
6,employment_duration
7,installment_rate_pct_income
8,personal_status_sex
9,other_debtors_guarantors


{'target_col': 'TARGET',
 'identifier_cols': ['applicant_id'],
 'target_label_cols': ['risk_class', 'risk_label'],
 'excluded_from_features': ['applicant_id',
  'risk_class',
  'risk_label',
  'TARGET'],
 'n_features': 20}

## 4. Target Balance Before Splitting

The target balance should be preserved across train, validation, and test sets. Because this is a classification problem, a stratified split is appropriate.

In [4]:
target_balance = pd.DataFrame(
    {
        "count": df[target_col].value_counts().sort_index(),
        "percentage": (df[target_col].value_counts(normalize=True).sort_index() * 100).round(2),
    }
)

target_balance.index = ["good_risk_0", "bad_risk_1"]
target_balance

,count,percentage
good_risk_0,700,70.0
bad_risk_1,300,30.0


## 5. Review Leakage Risks

Leakage happens when a feature contains information that would not be available at the time of a credit decision. The German Credit dataset is small and mostly application-time information, but target-derived fields must be excluded.

In [5]:
leakage_review = pd.DataFrame(
    [
        {
            "column": "risk_class",
            "risk": "Target-derived label",
            "decision": "Exclude from model features",
        },
        {
            "column": "risk_label",
            "risk": "Human-readable version of the target",
            "decision": "Exclude from model features",
        },
        {
            "column": "TARGET",
            "risk": "Modeling target",
            "decision": "Use only as y, never as X",
        },
        {
            "column": "applicant_id",
            "risk": "Identifier, not a predictive business feature",
            "decision": "Exclude from model features; keep for traceability",
        },
    ]
)

leakage_review

,column,risk,decision
0,risk_class,Target-derived label,Exclude from model features
1,risk_label,Human-readable version of the target,Exclude from model features
2,TARGET,Modeling target,"Use only as y, never as X"
3,applicant_id,"Identifier, not a predictive business feature",Exclude from model features; keep for traceabi...


## 6. Create Feature Matrix and Target Vector

Create `X` and `y` after excluding identifiers and target-derived columns. No preprocessing is fitted here.

In [6]:
X = df[feature_cols].copy()
y = df[target_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head()

X shape: (1000, 20)
y shape: (1000,)


,checking_account_status,duration_months,credit_history,purpose,credit_amount,savings_account,employment_duration,installment_rate_pct_income,personal_status_sex,other_debtors_guarantors,present_residence_years,property_type,age_years,other_installment_plans,housing,existing_credits_count,job_type,liable_people_count,telephone,foreign_worker
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,4,A121,67,A143,A152,2,A173,1,A192,A201
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,2,A121,22,A143,A152,1,A173,1,A191,A201
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,3,A121,49,A143,A152,1,A172,2,A191,A201
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,4,A122,45,A143,A153,1,A173,2,A191,A201
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,4,A124,53,A143,A153,2,A173,2,A191,A201


## 7. Create Stratified Train, Validation, and Test Splits

Use a 70/15/15 split. Stratification keeps the bad-risk rate stable across each split.

In [7]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_valid.shape, y_valid.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (700, 20) (700,)
Validation: (150, 20) (150,)
Test: (150, 20) (150,)


## 8. Validate Split Balance

Check that each split preserves the target distribution close to the original dataset.

In [8]:
def split_target_summary(name: str, target_series: pd.Series) -> dict:
    return {
        "split": name,
        "rows": int(len(target_series)),
        "bad_count": int(target_series.sum()),
        "bad_rate": round(float(target_series.mean()), 4),
        "good_count": int((target_series == 0).sum()),
        "good_rate": round(float((target_series == 0).mean()), 4),
    }

split_summary = pd.DataFrame(
    [
        split_target_summary("full", y),
        split_target_summary("train", y_train),
        split_target_summary("validation", y_valid),
        split_target_summary("test", y_test),
    ]
)

split_summary

,split,rows,bad_count,bad_rate,good_count,good_rate
0,full,1000,300,0.3,700,0.7
1,train,700,210,0.3,490,0.7
2,validation,150,45,0.3,105,0.7
3,test,150,45,0.3,105,0.7


## 9. Save Split Summary Artifact

Save the split configuration and target-balance summary so the decision is reproducible.

In [9]:
artifacts_dir = project_root / "artifacts" / "profiles"
artifacts_dir.mkdir(parents=True, exist_ok=True)

split_artifact = {
    "random_state": RANDOM_STATE,
    "split_strategy": "stratified_random_split",
    "split_ratio": {"train": 0.70, "validation": 0.15, "test": 0.15},
    "target_col": target_col,
    "excluded_from_features": exclude_from_features,
    "feature_count": len(feature_cols),
    "split_summary": split_summary.to_dict(orient="records"),
}

split_artifact_path = artifacts_dir / "split_strategy_summary.json"
with split_artifact_path.open("w") as f:
    json.dump(split_artifact, f, indent=2)

split_artifact_path

PosixPath('/Users/ememakpan/Documents/New project/applied_ai_economics_portfolio/project_01_ai_credit_risk_platform/artifacts/profiles/split_strategy_summary.json')

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 341, in dispatch_control
    await self.process_control(msg)
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 347, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/jupyter_client/session.py", line 994, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/opt/ana

## 10. Split Strategy Decision

A stratified random split is appropriate because this dataset does not contain a reliable application timestamp for a time-based split. The target distribution is preserved across train, validation, and test sets.

Target-derived columns and identifiers are excluded from model features. All fitted preprocessing steps must be learned on the training split only, then applied to validation and test sets.

Next notebook: `06_outlier_detection_and_preprocessing.ipynb`.